# Tools with MCP ⏰
The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using langchain-mcp-adapters.

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio

nest_asyncio.apply()

# Connect to the mcp-time server for timezone-aware operations
# This Go-based server provides tools for current time, relative time parsing,
# timezone conversion, duration arithmetic, and time comparison
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


Create an agent with the MCP-provided time tools

In [12]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="ollama:qwen2.5:3b",
    tools = mcp_tools,
    system_prompt="You are a helpful assitant"
)

Ask about the current time in anywhere in the world.

In [13]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What is the time in India right now"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the time in India right now
================================== Ai Message ==================================
Tool Calls:
  current_time (f5de58a2-1a96-495c-bf45-d81c3f2978c2)
 Call ID: f5de58a2-1a96-495c-bf45-d81c3f2978c2
  Args:
    timezone: Asia/Kolkata
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': '2026-07-10T22:50:26+05:30', 'id': 'lc_f91d64fd-ed4a-4c9c-9480-2377813fb2ce'}]
================================== Ai Message ==================================

The current time in India (Asia/Kolkata) is 2026-07-10T22:50:26+05:30.
